# **Smartphone Behaviour and Student Stress: A CRISP-DM Analysis of StudentLife**

## DBMS71008 Data Analytics | CA2 CRISP-DM Analytics Project | Daniel Courtney

## **Phase 1: Business Understanding**

### **1.1 Context and Problem**

University students face recurring periods of high academic pressure, and stress tends to rise as workload increases across the academic term. Heavy smartphone use and poor sleep are commonly suggested as contributing factors. However, student support services typically learn about difficulties only when students self-report them, after problems have escalated, with much of this stress remaining hidden from staff.

Smartphones offer a way to observe daily behaviour. This project investigates whether behavioural signals captured by smartphones are associated with students self-reported stress, using the *StudentLife dataset*. *(Wang et al., 2014)*


### **1.2 Stakeholders and Users**

| Stakeholder | Role | Interest / How they are affected |
| --- | --- | --- |
| StudentLife Participants | Supply data | Behaviour and stress data analysed for improved supports |
| Student Support Services | Primary User | Want insights into when students may need support, to guide outreach and resources |
| StudentLife Research Team | Data Owner | Set terms of use and attribution for the dataset |
| Researchers | Secondary Users | Interested in whether sensing can support wellbeing research |
| Data Protection Officer | Governance | Decide whether monitoring is acceptable, and under what consent, privacy, and data protection rules. |
| Current Students | Affected Party | Could benefit from earlier, better-timed support. But risks being wrongly flagged (false positive), missed when struggling (false negative), or feeling monitored |

### **1.3 Business or Practical Objective**

**Practical Objective:**

Help student support services understand which daily behaviours are associated with elevated stress, so they can decide when and how to offer proactive support.

### **1.4 Analytics Objective**

**Question**

To what extent can passively sensed smartphone usage, sleep, and physical activity distinguish days on which StudentLife participants report elevated stress from days on which they do not?


**Analytics Objective**

Build a binary classifier that distinguishes high stress from other student-days using daily phone use, sleep, and activity features, and evaluate whether it outperforms a majority-class baseline in recall and F1 for high-stress days on students not seen during training.

Recall is essential as it means support services missing a struggling student (false negative) is worse than an unnecessary check in (false positive) *(Adler et al., 2022)*. 


**Unit of Analysis**

One student on one calendar day.


**Sleep Measure**

Sleep duration is estimated from phone lock data (longest overnight lock period). Self-reported sleep is used only to validate this estimate.

### **1.5 Success Criteria**

**Technical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Beats the baseline | Recall and F1 for high-stress days exceed a majority-class baseline |
| Generalises to new students | Evaluated on students held out from training |

<br>

**Practical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Interpretable | Main patterns can be explained in plain language to non-technical support staff |
| Useful for the intended decision | Findings indicate which behaviours or periods relate to elevated stress, informing outreach timing |
| Appropriate use | Results describe patterns, not labels on individual students |
| Honest limitations | Sample size, single institution and 2013 data clearly stated |

<br>

Recall is prioritised over accuracy because, for support services, missing a genuinely stressed student *(a false negative)* is far more costly than an unnecessary check-in *(a false positive)*. F1 is reported alongside it so that precision is not ignored.

### **1.6 Constraints and Assumptions**

**Constraints**

| Type | Constraint |
| --- | --- |
| Time | Individual project, with final submission due 4 December 2026 |
| Data | Limited to the existing StudentLife data (48 students, one class, Spring 2013), no new data can be collected |
| Scope | Stress as the only outcome. Phone use, sleep and activity as the only feature areas. Location data excluded |

<br>

**Assumptions**

| Assumption | Why it matters |
| --- | --- |
| Self-reported stress reflects how stressed students actually felt | It is the target label. If it is inaccurate, the model learns noise |
| Activity is accurate enough for daily summaries | They are classifier outputs with known error |
| Students carried their study phone throughout the day | Days where they did not would understate behaviour |
| Student-days with sufficient data are representative of the term | If missing data is concentrated in stressful weeks, high-stress days are under-represented |
| The longest overnight phone-lock period approximates sleep duration | It overestimates sleep if the phone is untouched while awake, and underestimates it if the phone is checked during the night |

<br>

**Risks**

| Risk | Why it matters | Mitigation |
| --- | --- | --- |
| Sparse stress responses | Students did not answer every day, meaning fewer labelled rows | Drop or flag students with too few responses |
| Missing sensor data | Phone off, uncharged, or not carried | Exclude days with too little data |
| Ethics and Privacy | Behavioural and mental health data about real people | Frame the findings as patterns, not individual labels |
| Sleep proxy validity | The estimate may not reflect actual sleep, and is derived from the same stream as phone use | Validate against self-reported sleep where both exist. Check correlation with the phone-use feature, and fall back to self-reported sleep if the proxy proves unreliable |

### **1.7 Initial Project Plan**

**Proposed Technique:** 

Binary classification using a decision tree, compared against a majority class baseline and an alternative model, evaluated with recall and F1 on a student-level split.

<br>

**Tools:** 

Python, Jupyter Notebook, Github

<br>

**CRISP-DM Activities**

| Phase | Activities | Target Dates |
| --- | --- | --- |
| *Business Understanding* | Problem, stakeholders, objectives, success criteria, constraints | 09 Nov |
| *Data Understanding* | Feasibility check (stress responses per student, sensor coverage), data description, exploration and quality report, data dictionary | 10-16 Nov |
| *Data Preparation* | Recode stress scale, aggregate sensor data to student-days, join with stress labels, handle missing data, preparation log, before/after validation | 10-16 Nov |
| *Modelling* | Student-level train/test split, baseline, decision tree | 16-23 Nov |
| *Evaluation* | Metrics against success criteria, error analysis, limitations, bias and ethics | 23-30 Nov |
| *Deployment & submission* | Recommendations and monitoring, reflection, AI collaboration statement, final paper | 30 Nov -04 Dec |

**Checkpoint:** if the prepared student-day dataset is not complete by 20 November, scope will be reduced (fewer variables or students) rather than extending preparation, following lecturer feedback (10 Nov 2026).

*The plan is to be revised as the project develops, with changes and their reasoning recorded accordingly*

<br>

**Project Feedback**

| Date | Source | Feedback | How it was addressed |
| --- | --- | --- | --- |
| 10 Nov 2026 | Email, Andrew Shields | Continue with StudentLife,  combining JSON/CSV files into a student-day table is valid Data Preparation. Keep scope manageable, document every transformation step, and prioritise answering the question over data wrangling. Reduce variables or students rather than switching datasets if needed. | Kept StudentLife. Limited to four data streams, transformation steps recorded in the preparation log. |

<br>

**Project Decisions**

| Date | Type | Summary | Description |
| --- | --- | --- | --- |
| 10 Nov 2026 | Decision | Sleep measure | Chose a phone-lock sleep proxy over <br><br> (a) self-reported sleep, which covers only ~69% of stress days and shares self-report bias with the stress target, and <br><br> (b) a dark/charge proxy, which adds two extra streams beyond the time available. Self-reported sleep retained for validation only (cf. Abdullah et al., 2014). |

## **Phase 2: Data Understanding**

### **Environment Setup**

In [2]:
# data wrangling
import pandas as pd
import numpy as np
from pathlib import Path

# data viz
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# kaggle importing
import kagglehub

### **2.1 Initial Data Collection Report**

**2.1.1 Source and Permissions**

| Type | Details |
| --- | --- |
| Data Source | Kaggle dataset StudentLife https://www.kaggle.com/datasets/dartweichen/student-life?select=dataset |
| Owner / Publisher | StudentLife Dartmouth College (Wang et al., 2014) – https://StudentLife.cs.dartmouth.edu/ |
| Access Date | 29 October 2026 |
| Licence / Permission | No explicit licence is stated on the Kaggle page or the original Dartmouth dataset page. The Dartmouth team publicly releases the anonymised dataset for research, and asks that Wang et al. (2014) is cited if the dataset is used in a publication (StudentLife, n.d.). Use in this project is non-commercial and academic, the dataset is cited throughout, and raw data is not redistributed in the repository. |

<br>

**2.1.2 Collection Process**

| Type | Details |
| --- | --- |
| Collection Process | A StudentLife android app passively collected sensor based data such as activity, sleep, and location, and delivered EMA surveys. Pre and post term surveys were conducted via SurveyMonkey. |

<br>

**2.1.3 Population and Scope**

| Type | Details |
| --- | --- |
| Population | 48 Dartmouth students who completed the study |
| Time Period | Spring academic term 2013, for a period of 10 weeks. Approx late March to early June  |
| Geography | Dartmouth College, New Hampshire, USA. |
| Scope | **Dataset:** passive sensing inferences (activity, conversation, phone lock/charge, light, location), EMA responses (stress, mood, sleep, social and others, EMA and mood responses), pre/post mental health surveys, and education data (grades, deadlines) – 52.6 GB of sensing data in total (Wang et al., 2014). <br><br> **This project:** stress EMA (target), phone lock (used for both phone use and a sleep estimate), activity, and self-reported sleep (for validation only), with all other data excluded. |

<br>

**2.1.4 Exclusions and Limitations**

| Type | Details |
| --- | --- |
| Known Exclusions | Students who dropped out of the study or the class were removed (60 joined, 48 completed). |
| Aggregation | The sensor data are inferences from phone classifiers, not raw sensor readings. |
| Capping | None identified. |
| Sampling | Voluntary sample from one class, not random. |
| Access Limitations | Kaggle repost of the original |

<br>

**2.1.5 Files Acquired**

*AI-assisted:* The selection of files below was developed with the help of Claude (Anthropic, Opus 5.5). I provided my research question and the dataset link, and the tool identified which folders correspond to each part of the question, described their file formats and key fields, and suggested how they link together. I verified this by opening each file myself, confirming the field names and the stress answer coding against `EMA/EMA_definition.json`, and checking the activity codes against Wang et al. (2014). The final choice of files, including the sleep measure, is my own decision.

| Required from Question | File / folder | Contents | Key fields | How it relates |
| --- | --- | --- | --- | --- |
| Elevated stress (target) | `EMA/response/Stress/` | 49 JSON files, one per student (`Stress_uXX.json`), containing stress EMA responses | `level` (1–5, stored as text), `resp_time` (Unix seconds, UTC), `location`, student ID from filename | Target label, recoded to high / not-high stress and joined to features on student ID + date |
| Stress answer coding | `EMA/EMA_definition.json` | Wording and answer options for every EMA question | `name`, `question_id`, `question_text`, `options` | Confirms the stress scale is not ordered by severity (1 A little stressed … 5 Feeling great), so it must be recoded |
| Smartphone usage | `sensing/phonelock/` | 49 CSV files, one per student (`phonelock_uXX.csv`), listing periods the phone was locked | `start`, `end` (Unix seconds, UTC) | Aggregated to daily lock/unlock time per student; joined on student ID + date |
| Physical activity | `sensing/activity/` | 49 CSV files, one per student (`activity_uXX.csv`), with activity inferences every few seconds | `timestamp` (Unix seconds), `activity inference` (0 stationary, 1 walking, 2 running, 3 unknown; Wang et al., 2014) | Aggregated to daily active minutes; joined on student ID + date |
| Sleep (estimated) | `sensing/phonelock/` | Same files as smartphone usage | `start`, `end` (Unix seconds, UTC) | Longest lock period within an overnight window used as estimated sleep duration; joined on student ID + date |
| Sleep (validation only) | `EMA/response/Sleep/` | 49 JSON files, one per student, with sleep EMA responses | `hour` (codes 1–19 = <3 h to 12 h), `resp_time` | Not a model input; used to check the sleep estimate on days where both exist |

All files are linked by student ID (`u00`–`u59`, from the filename) and the local
date derived from the Unix timestamps.

### **2.2 Data Description Report**

This section describes the raw data as acquired, before any cleaning. Issues noticed here are recorded and carried forward to the *Data Quality Report* and addressed in *Data Preparation*.

In [3]:
# Using kagglehub so that the dataset will download when this code is run.
# https://pypi.org/project/kagglehub/
path = Path(kagglehub.dataset_download("dartweichen/student-life", output_dir="../data/raw/student-life"))
DATA = next(path.rglob("EMA_definition.json")).parent.parent
print(DATA, DATA.exists())

100%|██████████| 390M/390M [00:12<00:00, 32.6MB/s] 

Extracting files...


..\data\raw\student-life\dataset True


In [4]:
"""
Converting the JSON/CSV files into a DataFrame
------------------------------------------
Finds JSON files, reads each into a table, tags it with an uid, and stacks them.
Finds CSVs, reads, tags, and stacks.

https://docs.python.org/3/library/json.html
https://docs.python.org/3/library/pathlib.html
https://pandas.pydata.org/docs/user_guide/merging.html
https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
https://pandas.pydata.org/docs/user_guide/timeseries.html
https://pandas.pydata.org/docs/user_guide/10min.html
"""

import json

def load_EMA(folder):
    """Load all EMA JSON files in a folder into one raw DataFrame."""
    frames = []

    for file in sorted((DATA / "EMA/response" / folder).glob("*.json")):
        df = pd.DataFrame(json.load(open(file)))
        df["uid"] = file.stem.split("_")[1]   # e.g. Stress_u00 -> u00
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def load_sensing(folder):
    """Load all sensing CSV files in a folder into one raw DataFrame."""
    frames = []

    for file in sorted((DATA / "sensing" / folder).glob("*.csv")):
        df = pd.read_csv(file)
        df["uid"] = file.stem.split("_")[1]   # e.g. phonelock_u00 -> u00
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

stress = load_EMA("Stress")
sleep = load_EMA("Sleep") # validation
phonelock = load_sensing("phonelock") # phone use and sleep proxy
activity = load_sensing("activity")

**2.2.1 Dataset Dimensions and Unit of Analysis**

The four raw streams were loaded from 49 per-student files each. Their structure differs considerably:

| Stream | Format | One row represents |
| --- | --- | --- |
| Stress | JSON | One stress EMA record (incl junk entries with no stress level). |
| Sleep | JSON | One sleep EMA record (validation).
| Phone Lock | CSV | One period during which the phone was locked. |
| Activity | CSV | One activity inference, recorded every few seconds. |

Column counts include the uid column added during loading. All streams cover the same 49 students, although 48 completed the study (Wang et al., 2014).

<br>

**Unit of Analysis**

One student on one calendar day.

<br>

**Analysis**

Stress and sleep are recorded per response, phone lock as time intervals, and activity as near-continuous readings. Converting each stream to one row per student per day, and joining them on student ID and date, is therefore the central Data Preparation task.

<br>

**Carry into Data Quality Report**

- 49 students have files, but only 48 completed the study
- Stress responses extend beyond the sensing period
- Timestamps are Unix seconds in UTC and must be converted

In [5]:
# Validate that everything works - show number of columns and rows

# The raw tables to summarise
streams = {
    "Stress": stress,
    "Sleep": sleep,
    "Phone Lock": phonelock,
    "Activity": activity,
}

# Record the size of each table
rows = []

for name, df in streams.items():
    rows.append({
        "Description": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Students": df["uid"].nunique(),
    })

summary = pd.DataFrame(rows)
summary

,Description,Rows,Columns,Students
0,Stress,2408,5,49
1,Sleep,1644,7,49
2,Phone Lock,9275,3,49
3,Activity,22842191,3,49


**2.2.2 Variable Names, Meanings, Data Types, Units, and Valid Ranges**

The cells below inspect each streams columns and data types, the official answer coding for the EMA questions, and the values actually observed in the data.

| Brief Requirement | Cell |
| --- | --- |
| Variable names and data types | Data types for each stream (`dtypes`) |
| Units and record structure | First rows of each stream (`head`) |
| Meanings | Official question wording and answer coding (`EMA_definition.json`) |
| Valid ranges (target) | Observed stress `level` values |
| Valid ranges (validation) | Observed sleep `hour` codes |
| Meanings and valid ranges (input) | Observed activity codes |
| Valid ranges (input) | Phone lock check: periods where end ≤ start |

In [6]:
# List every column in each stream with its data type
for name, df in streams.items():
    print(f"Stream: {name}")
    display(df.dtypes.to_frame("Data type"))

Stream: Stress


,Data type
null,str
resp_time,int64
level,str
location,str
uid,str


Stream: Sleep


,Data type
null,str
resp_time,int64
hour,str
location,str
rate,str
social,str
uid,str


Stream: Phone Lock


,Data type
start,int64
end,int64
uid,str


Stream: Activity


,Data type
timestamp,int64
activity inference,int64
uid,str


In [7]:
# Show the first rows of each stream
for df in [stress, sleep, phonelock, activity]:
    display(df.head(3))

,null,resp_time,level,location,uid
0,"43.75908069,-72.32885314",1364114401,NaN,NaN,u00
1,"43.75908069,-72.32885314",1364114458,NaN,NaN,u00
2,"43.70677151,-72.28746626",1364177805,NaN,NaN,u00


,null,resp_time,hour,location,rate,social,uid
0,"43.75908069,-72.32885314",1364114760,NaN,NaN,NaN,NaN,u00
1,8,1364114765,NaN,NaN,NaN,NaN,u00
2,1,1364114453,NaN,NaN,NaN,NaN,u00


,start,end,uid
0,1364358992,1364364807,u00
1,1364365565,1364381933,u00
2,1364386333,1364391008,u00


,timestamp,activity inference,uid
0,1364356801,0,u00
1,1364356804,0,u00
2,1364356807,0,u00


In [8]:
# Read the question wording and answer options for the stress and sleep EMAs
definitions = json.load(open(DATA / "EMA/EMA_definition.json"))

rows = []

for survey in definitions:
    if survey["name"] in ("Stress", "Sleep"):
        for question in survey["questions"]:
            if question["options"]:
                rows.append({
                    "Survey": survey["name"], 
                    "Field": question["question_id"],
                    "Question": question["question_text"],
                    "Option": question["options"]
                })

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,Survey,Field,Question,Option
0,Stress,level,"Right now, I am...","[1]A little stressed, [2]Definitely stressed, [3]Stressed out, [4]Feeling good, [5]Feeling great,"
1,Sleep,hour,How many hours did you sleep last night?,"[1]<3, [2]3.5, [3]4, [4]4.5, [5]5, [6]5.5, [7]6, [8]6.5, [9]7, [10]7.5, [11]8, [12]8.5, [13]9, [14]9.5, [15]10, [16]10.5, [17]11, [18]11.5, [19]12,"
2,Sleep,rate,How would rate your overall sleep last night?,"[1]Very good, [2]Fairly good, [3]Fairly bad, [4]Very bad,"
3,Sleep,social,"How often did you have trouble staying awake yesterday while in class, eating meals or engaging in social activity?","[1]None, [2]Once, [3]Twice, [4]Three or more times,"


In [9]:
# Stress level: what values actually appear (including missing)
stress["level"].value_counts(dropna=False)

level
1      974
4      457
2      347
3      266
NaN    241
5      123
Name: count, dtype: int64

In [10]:
# Sleep hours: observed answer codes
pd.to_numeric(sleep["hour"]).value_counts(dropna=False).sort_index()

hour
0.0      12
1.0      26
2.0      12
3.0      35
4.0      49
5.0     110
6.0     207
7.0     289
8.0     358
9.0     140
10.0     67
11.0     28
12.0     21
13.0     13
14.0      5
15.0     12
16.0      2
17.0      2
19.0      2
NaN     254
Name: count, dtype: int64

In [11]:
# Activity codes (note: column name may contain a leading space)
activity_col = [c for c in activity.columns if "activity" in c][0]
print(repr(activity_col))
activity[activity_col].value_counts()

' activity inference'


 activity inference
0    21022995
1      972376
3      608598
2      238222
Name: count, dtype: int64

In [12]:
# Phone lock: check that every period ends after it starts
invalid_periods = (phonelock["end"] <= phonelock["start"]).sum()
print("Lock periods where end <= start:", invalid_periods)

Lock periods where end <= start: 0


**Observations**

- **Data Types:** `level` and `hour` are stored as text(str), not numbers. They must be converted before analysis. Timestamps are integers (Unix Seconds).

- **Stress Level:** values 1-5 match the definition file, but the scale is not ordered by severity (1 being "a little stressed" to 5 being "feeling great"), so it must be recoded. 241 records have no level.

- **Sleep Hour:** valid codes are 1-19, but 12 records contain an invalid code 0 and 254 are missing. Code 18 never appears.

- **Activity:** codes 0-3 as expected. Stationary (0) accounts for 92% of readings (21,022,995 / 22,842,191).

- **Phone Lock:** no periods end at or before their start, so lock durations can be calculated safely.

- **Contents of null:** in stress, 109 values are GPS coordinates and 131 are othervalues (mainly the digits 1–4 and "Unknown"); in sleep, 63 are coordinates and 191 are other values (mainly 1, 8 and 3). These digits may be answers stored under the wrong key, but this cannot be confirmed, so the records are treated as unusable.

<br>

These issues are to be carried forward to the data quality report.

**2.2.3 Identifiers, Candidate Inputs, and Potential Target or Outcome**

*AI-assisted: table structure and wording drafted with Claude, each row checked against the outputs in 2.2.2.*

| Role | Variable(s) | Source | Notes |
| --- | --- | --- | --- |
| Identifier | `uid` | Filename | Whose record it is; links all four streams |
| Identifier (time) | `resp_time`, `start`, `end`, `timestamp` | All streams | When it happened; converted to a local date in Phase 3 |
| Potential target | `level` | Stress EMA | Recoded to high / not-high stress in Phase 3 |
| Candidate input | `start`, `end` | Phone Lock | Daily unlocked time (phone use) |
| Candidate input | `start`, `end` | Phone Lock | Longest overnight lock period (sleep estimate) |
| Candidate input | `activity inference` | Activity | Daily active minutes |
| Other: validation | `hour` | Sleep EMA | Checks the sleep estimate; not a model input |
| Other: not used | `location`, `null`, `rate`, `social` | EMA files | Privacy, junk field, or outside scope |

**2.2.4 Data Dictionary or Equivalent Table**

The data dictionary combines the findings of 2.2.2 and 2.2.3 into one reference table. Factual properties (data type, missing values, distinct values and date ranges) are calculated directly from the data below. Meanings and valid ranges are taken from EMA_definition.json and Wang et al. (2014), and roles from 2.2.3.

In [13]:
# Build the factual part of the data dictionary from the data itself
rows = []

for name, df in streams.items():
    for column in df.columns:
        rows.append({
            "Variable": column.strip(),
            "Source": name,
            "Raw type": str(df[column].dtype),
            "Missing": int(df[column].isna().sum()),
            "Unique values": df[column].nunique(),
        })

pd.DataFrame(rows).set_index(["Source", "Variable"])

Raw type  Missing  Unique values
Source     Variable                                           
Stress     null                    str     2168             79
           resp_time             int64        0           2363
           level                   str      241              5
           location                str      241           1583
           uid                     str        0             49
Sleep      null                    str     1390             61
           resp_time             int64        0           1617
           hour                    str      254             19
           location                str      256           1193
           rate                    str      254              4
           social                  str      254              4
           uid                     str        0             49
Phone Lock start                 int64        0           9266
           end                   int64        0           9233
           uid                     str        0             49
Activity   timestamp             int64        0        5573234
           activity inference    int64        0              4
           uid                     str        0             49

In [14]:
# Check whether records with a 'null' value are the same records missing an answer
for name, df, answer in [("Stress", stress, "level"), ("Sleep", sleep, "hour")]:
    has_null = df["null"].notna()
    no_answer = df[answer].isna()

    print(f"\n{name}:")
    print("  null filled, answer missing: ", (has_null & no_answer).sum())
    print("  null filled, answer present: ", (has_null & ~no_answer).sum())
    print("  null empty,  answer missing: ", (~has_null & no_answer).sum())


Stress:
  null filled, answer missing:  240
  null filled, answer present:  0
  null empty,  answer missing:  1

Sleep:
  null filled, answer missing:  254
  null filled, answer present:  0
  null empty,  answer missing:  0


In [15]:
# Classify the contents of the 'null' field: coordinates (contain a comma) or other values
for name, df in [("Stress", stress), ("Sleep", sleep)]:
    values = df["null"].dropna()
    is_coordinates = values.str.contains(",")

    print(f"{name}: {len(values)} populated")
    print("  look like coordinates:", is_coordinates.sum())
    print("  other values:         ", (~is_coordinates).sum())
    print("  most common other values:", values[~is_coordinates].value_counts().head(5).to_dict())

Stress: 240 populated
  look like coordinates: 109
  other values:          131
  most common other values: {'3': 44, '1': 39, '2': 22, '4': 15, 'Unknown': 9}
Sleep: 254 populated
  look like coordinates: 63
  other values:          191
  most common other values: {'1': 70, '8': 35, '3': 32, '2': 15, '7': 15}


In [16]:
# Convert each stream's time field from Unix seconds to dates to show the period covered
time_fields = {
    "Stress": (stress, "resp_time"),
    "Sleep": (sleep, "resp_time"),
    "Phone Lock": (phonelock, "start"),
    "Activity": (activity, "timestamp"),
}

rows = []

for name, (df, column) in time_fields.items():
    rows.append({
        "Source": name,
        "First record (UTC)": pd.to_datetime(df[column].min(), unit="s"),
        "Last record (UTC)": pd.to_datetime(df[column].max(), unit="s"),
    })

pd.DataFrame(rows).set_index("Source")

,First record (UTC),Last record (UTC)
Source,,
Stress,2013-03-24 08:40:01,2013-08-16 00:56:08
Sleep,2013-03-24 08:39:25,2013-06-07 23:46:41
Phone Lock,2013-03-26 23:00:41,2013-06-01 02:40:56
Activity,2013-03-27 04:00:00,2013-06-01 03:59:59


**Observations**

*The output confirms the findings in 2.2.2 and adds three observations for the Data Quality Report:*

- **Junk Records Confirmed:** in stress, all 240 records with a null value are missing level, and 1 futher record has no response data at all. In sleep, all 254 with a null value are missing hour. No record has both a null value and a valid answer.

- **Repeated Timestamps:** stress resp_time (45) and phone lock start (9) and end (42) contain repeated values. Within-student duplicates are checked in 2.4.

- **Date ranges**: sensing data covers 26–27 March to 1 June 2013 (UTC), while stress responses continue until 16 August and sleep until 7 June. Records outside the sensing period will need filtering.

*AI-assisted: table structure (headings) drafted with Claude.*

**How to read the tables**

| Column | Where the value comes from |
| --- | --- |
| Raw type, Missing, Unique | The properties cell at the top of 2.2.4 (calculated from the data) |
| Meaning, Valid range | `EMA_definition.json` (2.2.2) for EMA fields, Wang et al. (2014) for activity codes |
| Units | First rows of each stream (2.2.2): 10-digit integers are Unix seconds |
| Observed in raw data | The value checks in 2.2.2 and the junk-record, `null` contents and date-range checks in 2.2.4 |
| Role | 2.2.3 |

<br>

**Stress**
| Variable | Source | Meaning | Raw type | Units | Valid range | Missing | Unique | Observed in raw data | Role |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **null** | Stress | Unlabelled field | str | - | - | 2168 | 79 | Populated only in 240 junk records: 109 coordinates, 131 | Not used |
| **resp_time** | Stress | Time the response was submitted | int64 | Unix seconds | Sensing period | 0 | 2363 | 45 repeated values, with responses continuing to August 16th, 2013 | Identifer (time) |
| **level** | Stress | "Right now I am..." | str | Category | 1-5 (1 being a little stressed and 5 feeling great) | 241 | 5 | Scale is not ordered by severity, 1 record has no response data at all | Potential target |
| **location** | Stress | GPS coordinates at response | str | Latitude, longitude | - | 241 | 1583 | Missing in the same records as level | Not used due to privacy |
| **uid** | Stress | Student identifier | str | - | u00-u59 | 0 | 49 | Added from filename | Identifier |

The stress table contains 2,408 records from 49 students. Of these, 241 have no `level`: 240 are junk records in which the app saved a value under an unlabelled `null` field instead (109 GPS coordinates and 131 stray values, mainly the digits 1–4 and "Unknown"), and 1 record contains no response data at all. 

This leaves 2,167 usable stress responses. Responses are coded 1–5, but the scale is not ordered by severity (1 = "A little stressed", 5 = "Feeling great"), so it must be recoded before a high-stress threshold can be set. 45 response times are repeated, and some responses were submitted as late as 16 August 2013, well after sensing ended on 1 June, so these will need filtering.

<br>

**Sleep**

| Variable | Source | Meaning | Raw type | Units | Valid range | Missing | Unique | Observed in raw data | Role |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **null** | Sleep | Unlabelled field | str | - | - | 1390 | 61 | Populated only in 254 junk records: 63 coordinates, 191 other values | Not used |
| **resp_time** | Sleep | Time the response was submitted | int64 | Unix seconds | Sensing period | 0 | 1617 | Responses continue to June 7, 2013 | Identifer (time) |
| **hour** | Sleep | "How many hours did you sleep last night?" | str | Category | 1-19 (<3hr to 12hr>) | 254 | 19 | 12 invalid 0s, code 18 is unused | Validation |
| **location** | Sleep | GPS coordinates at response | str | Latitude, longitude | – | 256 | 1,193 | 2 valid answers without a location | Not used (privacy) |
| **rate** | Sleep | Overall sleep quality last night | str | Category | 1–4 (Very good … Very bad) | 254 | 4 | Missing only in junk records | Not used |
| **social** | Sleep | Trouble staying awake during the day | str | Category | 1–4 (None … Three or more times) | 254 | 4 | Missing only in junk records | Not used |
| **uid** | Sleep | Student identifier | str | – | u00–u59 | 0 | 49 | Added from filename | Identifier |

The sleep table contains 1,644 records from 49 students. 254 are junk records with a value in the `null` field (63 GPS coordinates and 191 stray values, mainly 1, 8 and 3) and no `hour`, `rate` or `social` answer. Of the remaining 1,390 answers, 12 contain the code 0, which is not defined in `EMA_definition.json`, leaving 1,378 valid sleep-duration answers. 

Codes run from 1 (<3 hours) to 19 (12 hours) in half-hour steps. Code 18 was never selected. Two valid answers have no location, which does not matter as location is not used. Self-reported sleep is used only to validate the phone-lock sleep estimate, not as a model input.

<br>

**Phone Lock**

| Variable | Source | Meaning | Raw type | Units | Valid range | Missing | Unique | Observed in raw data | Role |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **start** | Phone Lock | Start of a lock period | int64` | Unix seconds (UTC) | Sensing period | 0 | 9,266 | 9 repeated values; 26 Mar – 1 Jun 2013 | Candidate input (derived) |
| **end** | Phone Lock | End of a lock period | int64 | Unix seconds (UTC) | Later than start | 0 | 9,233 | 42 repeated values; 0 periods end at or before their start | Candidate input (derived) |
| **uid** | Phone Lock | Student identifier | str | – | u00–u59 | 0 | 49 | Added from filename | Identifier |

The phone lock table contains 9,275 lock periods from 49 students, recorded between 26 March and 1 June 2013 (UTC). Each row gives the start and end of a period during which the phone was locked. There are no missing values, and no period ends at or before its start, so durations can be calculated safely. Phone use is not recorded directly: unlocked time must be inferred from the gaps between lock periods, and the sleep estimate from the longest overnight lock period.

On average there are only about 189 lock periods per student, or roughly 3 per day over the term, which is far fewer than the number of times a phone is typically locked. This suggests the stream may record only longer locked periods. the distribution of lock durations will be examined in the Data Exploration Report (2.3). Repeated start (9) and end (42) times may reflect different students sharing a timestamp and are checked in 2.4.

<br>

**Activity**

| Variable | Source | Meaning | Raw type | Units | Valid range | Missing | Unique | Observed in raw data | Role |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **timestamp** | Activity | Time of the inference | int64 | Unix seconds (UTC) | Sensing period | 0 | 5,573,234 | Readings every few seconds; 27 Mar – 1 Jun 2013 | Identifier (time) |
| **activity inference** | Activity | Inferred activity | int64 | Category | 0 stationary, 1 walking, 2 running, 3 unknown | 0 | 4 | 92% stationary, column name has a leading space | Candidate input (derived) |
| **uid** | Activity | Student identifier | str | – | u00–u59 | 0 | 49 | Added from filename | Identifier |

The activity table contains 22,842,191 activity inferences from 49 students, recorded every few seconds. The first reading is at 04:00 UTC on 27 March and the last at 03:59:59 UTC on 1 June 2013, which corresponds to midnight at the start of 27 March to the end of 31 May in local (US Eastern) time. There are no missing values and only the four documented codes appear: stationary (0) accounts for 92.0% of readings, walking (1) 4.3%, running (2) 1.0% and unknown (3) 2.7%. 

Because there are no blank values, any gaps in activity data appear as missing time periods rather than missing values, and must be identified by checking coverage per student-day. The column name contains a leading space (`' activity inference'`), which will be removed in Data Preparation.


### **2.3 Data Exploration Report**

### **2.4 Data Quality Report**